# Async optogenetic demo with napari + ExperimentStatusWidget

Same virtual-microscope optogenetic pipeline as `demo_sim_optogenetic.ipynb`, but:

- A live **napari** viewer with the **napari-micromanager** GUI is opened first, so frames stream in as they are acquired.
- The new **`ExperimentStatusWidget`** is docked into the viewer. It mirrors the controller's current `RunHandle`: state, current FOV / event index, frames received, lag, elapsed time, and a **Stop** button.
- `ctrl.run_experiment(events)` is now **non-blocking** and returns a `RunHandle`. The notebook returns control to the kernel immediately so you can interact with the viewer, poll status, cancel the run, or queue a `continue_experiment(...)`.

Requirements:

```
uv sync --extra virtual-microscope
```

Run cells one at a time in a Jupyter kernel that has Qt running (start the Jupyter kernel from a terminal so the napari window can pop up).


## 1. Start the virtual microscope

In [1]:
from virtual_microscope.backends.optogenetic import setup_optogenetic
from faro.microscope.simulation import UniMMCoreSimulation
import faro.core.utils as utils

core, sim = setup_optogenetic()
utils.print_configs(core)

mic = UniMMCoreSimulation(mmc=core)
mic.init_scope()  # detect SLM device for optogenetic stimulation

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Config Groups
└── Channel
    ├── DAPI
    ├── membrane
    └── phase-contrast

## 2. Open napari + napari-micromanager

We attach the simulated `core` to a `napari-micromanager` `MainWindow` so frames stream into the viewer as they are acquired.

In [2]:
import napari
from napari_micromanager import MainWindow

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = core
viewer.window.add_dock_widget(mm_wdg, name="napari-micromanager")

## 3. Build the pipeline (segmentation + tracking + features + stimulation)

Same components as the original demo: Otsu/watershed segmentation, trackpy linker, simple region-props feature extractor, and the `StimUpDown` stimulator that pushes even-particle cells up and odd-particle cells down.

In [3]:
import numpy as np
from skimage.filters import gaussian, threshold_otsu
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import label, regionprops
from skimage.morphology import disk, dilation
from scipy import ndimage
import skimage.measure
import pandas as pd

from faro.segmentation.base import Segmentator
from faro.feature_extraction.base import FeatureExtractor
from faro.stimulation.base import StimWithPipeline
from faro.tracking.trackpy import TrackerTrackpy
from faro.core.data_structures import SegmentationMethod


class WatershedSegmentator(Segmentator):
    """Gaussian blur -> Otsu -> distance transform -> watershed."""

    def segment(self, image: np.ndarray) -> np.ndarray:
        blurred = gaussian(image, sigma=3)
        thresh = threshold_otsu(blurred)
        binary = blurred > thresh
        distance = ndimage.distance_transform_edt(binary)
        coords = peak_local_max(distance, min_distance=20, labels=binary)
        markers = np.zeros(distance.shape, dtype=bool)
        markers[tuple(coords.T)] = True
        markers = label(markers)
        return watershed(-distance, markers, mask=binary)


class SimpleFE(FeatureExtractor):
    def __init__(self, used_mask):
        self.used_mask = used_mask
        super().__init__()

    def extract_features(self, labels, image, df_tracked=None, metadata=None):
        table = skimage.measure.regionprops_table(
            labels[self.used_mask], properties=["label", "area"]
        )
        return pd.DataFrame.from_dict(table), None


class StimUpDown(StimWithPipeline):
    """Even particles -> illuminate top edge; odd particles -> bottom edge."""

    def __init__(self, fraction=0.2):
        self.fraction = fraction

    def get_stim_mask(self, label_images, metadata=None, img=None, tracks=None):
        labels = label_images["labels"]
        stim_mask = np.zeros(labels.shape, dtype=np.uint8)
        selem = disk(3)

        if tracks is None or tracks.empty:
            return stim_mask, None

        current = tracks[tracks["timestep"] == tracks["timestep"].max()]
        label_to_particle = dict(zip(current["label"], current["particle"]))

        for prop in regionprops(labels):
            minr, minc, maxr, maxc = prop.bbox
            pid = label_to_particle.get(prop.label, 0)

            if pid % 2 == 0:
                y_cutoff = minr + self.fraction * (maxr - minr)
                select = lambda rows, c=y_cutoff: rows < c
            else:
                y_cutoff = maxr - self.fraction * (maxr - minr)
                select = lambda rows, c=y_cutoff: rows > c

            cell_mask = labels == prop.label
            rows, cols = np.where(cell_mask)
            edge_pixels = select(rows)
            if not edge_pixels.any():
                continue

            local = np.zeros_like(labels, dtype=np.uint8)
            local[rows[edge_pixels], cols[edge_pixels]] = 1
            local = dilation(local, footprint=selem)
            stim_mask = np.maximum(stim_mask, local)

        return stim_mask, None


segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=WatershedSegmentator(),
        use_channel=0,
        save_tracked=True,
    )
]

tracker = TrackerTrackpy(search_range=15)
feature_extractor = SimpleFE("labels")
stimulator = StimUpDown(fraction=0.2)

## 4. Assemble the pipeline + controller

In [4]:
import os, shutil, tempfile
from faro.core.pipeline import ImageProcessingPipeline
from faro.core.controller import Controller
from faro.core.writers import TiffWriter

path = tempfile.mkdtemp(prefix="napari_async_optogenetic_")
if os.path.exists(path):
    shutil.rmtree(path)

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
)

writer = TiffWriter(storage_path=path)
ctrl = Controller(mic, pipeline, writer=writer)
print(f"Storage path: {path}")

Directory /var/folders/zy/d2yp5vws25l6vkr2g5l4t39c0000gn/T/napari_async_optogenetic_yg4xg969/tracks created 
Storage path: /var/folders/zy/d2yp5vws25l6vkr2g5l4t39c0000gn/T/napari_async_optogenetic_yg4xg969


## 5. Dock the `ExperimentStatusWidget`

The widget subscribes to `ctrl.runStarted`, so it automatically re-binds whenever a new run begins. The **Stop** button calls `handle.cancel()` on the currently bound run.

In [5]:
from faro.widgets import ExperimentStatusWidget

status_widget = ExperimentStatusWidget(ctrl)
viewer.window.add_dock_widget(status_widget, name="experiment status", area="right")

## 6. Build a multi-phase event sequence

Baseline → stimulation → recovery, all on a single FOV. Phases are joined with `combine(..., axis="t")`, which offsets each subsequent sequence's `t` indices and `min_start_time` past the previous one. Slow enough that you can comfortably interact with the napari viewer while it runs.

In [6]:
from faro.core.data_structures import RTMSequence, combine, wait
from faro.core.utils import events_to_dataframe

n_baseline = 5
n_stim = 5
n_recovery = 5
interval_s = 1  # seconds between frames; raise if your machine struggles
wait_s = 10      # pre-baseline settle + post-stim recovery pauses

baseline = RTMSequence(
    time_plan={"interval": interval_s, "loops": n_baseline},
    stage_positions=[(0.0, 0.0, 0.0),(20,20,0.0)],
    channels=[{"config": "phase-contrast", "exposure": 50}],
    rtm_metadata={"phase": "baseline"},
)

stim_phase = RTMSequence(
    time_plan={"interval": interval_s, "loops": n_stim},
    stage_positions=[(0.0, 0.0, 0.0),(20,20,0.0)],
    channels=[{"config": "phase-contrast", "exposure": 50}],
    stim_channels=[{"config": "phase-contrast", "exposure": 50}],
    stim_frames=range(n_stim),
    rtm_metadata={"phase": "stimulation"},
)

recovery = RTMSequence(
    time_plan={"interval": interval_s, "loops": n_recovery},
    stage_positions=[(0.0, 0.0, 0.0),(20,20,0.0)],
    channels=[{"config": "phase-contrast", "exposure": 50}],
    rtm_metadata={"phase": "recovery"},
)

events = combine(wait(wait_s), baseline, stim_phase, wait(wait_s), recovery, axis="t")
df_events = events_to_dataframe(events)
print(
    f"{len(events)} events: {wait_s}s wait + {n_baseline} baseline + {n_stim} stim "
    f"+ {wait_s}s wait + {n_recovery} recovery"
)
df_events.head()

32 events: 10s wait + 5 baseline + 5 stim + 10s wait + 5 recovery


,fov,timestep,time,x_pos,y_pos,z_pos,channels,stim_channels,ref_channels,stim,ref,phase,stim_power,stim_exposure
0,0,0,0.0,NaN,NaN,NaN,(),(),(),False,False,NaN,NaN,NaN
1,0,0,11.0,0.0,0.0,0.0,"({'config': 'phase-contrast', 'exposure': 50.0...",(),(),False,False,baseline,NaN,NaN
2,0,0,21.0,NaN,NaN,NaN,(),(),(),False,False,NaN,NaN,NaN
3,1,0,11.0,20.0,20.0,0.0,"({'config': 'phase-contrast', 'exposure': 50.0...",(),(),False,False,baseline,NaN,NaN
4,0,1,12.0,0.0,0.0,0.0,"({'config': 'phase-contrast', 'exposure': 50.0...",(),(),False,False,baseline,NaN,NaN


## 7. Launch the run (non-blocking)

`run_experiment` now returns a `RunHandle` immediately. The MDA feed loop is on a worker thread; the kernel is free.

Watch the **napari viewer** for live frames and the **Experiment status** dock for state / FOV / event index / lag.  Try clicking **Stop** mid-run — the worker exits at the next event boundary.

In [8]:
handle = ctrl.run_experiment(events, stim_mode="previous")
print(f"run started, handle.is_running()={handle.is_running()}")
print(f"current status: {handle.status().state}")

run started, handle.is_running()=True
current status: running


/opt/homebrew/Cellar/python@3.12/3.12.12/Frameworks/Python.framework/Versions/3.12/lib/python3.12/threading.py:1012: UserWarning: FOV 0 position changed: (0.0, 0.0, 0.0) -> (None, None, None). Tracking continuity may be broken.
  self._target(*self._args, **self._kwargs)
/opt/homebrew/Cellar/python@3.12/3.12.12/Frameworks/Python.framework/Versions/3.12/lib/python3.12/threading.py:1012: UserWarning: FOV 0 position changed: (None, None, None) -> (0.0, 0.0, 0.0). Tracking continuity may be broken.
  self._target(*self._args, **self._kwargs)


### Poll the status from the kernel

Re-run the cell below as often as you like while the experiment is running.

In [8]:
s = handle.status()
print(f"state           : {s.state}")
print(f"events consumed : {s.n_events_consumed} / {s.n_events_total}")
print(f"frames received : {s.n_frames_received}")
print(f"current FOV     : {s.current_fov}")
print(f"current event   : {s.current_event_index}")
print(f"lag (ms)        : {s.lag_ms}")
print(f"bg errors       : {len(s.background_errors)}")
print(f"fatal error     : {s.fatal_error!r}")

state           : waiting
events consumed : 1 / 32
frames received : 0
current FOV     : None
current event   : {}
lag (ms)        : None
bg errors       : 0
fatal error     : None


### Subscribe to live updates from the kernel

Anything you do here is in *addition* to the dock widget — the widget already listens. Useful if you want to pipe status into your own logging.

In [9]:
def _print_status(status):
    print(f"[notify] state={status.state} consumed={status.n_events_consumed}"
          f"/{status.n_events_total} frames={status.n_frames_received}"
          f" lag_ms={status.lag_ms}")

handle.statusChanged.connect(_print_status)

<function __main__._print_status(status)>

### Cancel from the kernel (alternative to the Stop button)

In [10]:
# handle.cancel()  # uncomment to abort

### Block until the run finishes

Equivalent to the old synchronous behavior. Re-raises any worker-side `fatal_error`.

In [9]:
final_status = handle.wait()
print(f"final state     : {final_status.state}")
print(f"frames received : {final_status.n_frames_received}")
if final_status.fatal_error is not None:
    print(f"fatal           : {final_status.fatal_error!r}")

final state     : done
frames received : 80


## 8. Continue the experiment (optional)

`ctrl.continue_experiment(...)` reuses the existing Analyzer and tracking state. Same async semantics: returns a fresh `RunHandle`, the widget re-binds automatically via `runStarted`.

In [ ]:
extra = RTMSequence(
    time_plan={"interval": interval_s, "loops": 10},
    stage_positions=[(0.0, 0.0, 0.0)],
    channels=[{"config": "phase-contrast", "exposure": 50}],
    rtm_metadata={"phase": "extra-recovery"},
)

handle2 = ctrl.continue_experiment(list(extra), stim_mode="previous")
print(f"continuation started, handle2.is_running()={handle2.is_running()}")

In [ ]:
handle2.wait()
print(handle2.status().state)

## 9. Finish + tear down

In [ ]:
mic.post_experiment()
ctrl.finish_experiment()
print(f"All data written to: {path}")